# Exercice 2 : le moteur de workflow

**Mission :** exécuter un **playbook écrit en YAML**, étape par étape, avec des données qui passent d'une étape à l'autre.

```
 alerte ──► API (webhook) ──► moteur de workflow ──► exécuteur ──► actions
                               ex. 2  ◄── vous êtes ici            ex. 1 ✅
```

Dans un vrai SOAR, les analystes **n'écrivent pas de Python** pour chaque nouvel incident : ils écrivent (ou dessinent) un **playbook**, et le moteur l'exécute. Le playbook est une **donnée**, pas du code. Notre format est celui de **Tracecat**, en plus petit.

**Ouvrez `playbooks/connexion-suspecte.yaml` et `minisoar/moteur.py` dans VS Code.**

In [1]:
from banc import scenario, environnement as env
from minisoar import moteur

alertes = scenario.construire()
print(open("playbooks/connexion-suspecte.yaml").read())

title: Connexion suspecte
description: Une connexion réussie après des échecs (règle R2), ou un essai sur un compte leurre (règle R3).
actions:
  - ref: bilan
    action: journaux.connexions
    args:
      ip: ${{ TRIGGER.ip }}

  - ref: enrichir
    action: ip.enrichir
    args:
      ip: ${{ TRIGGER.ip }}
    retry_policy: {max_attempts: 3, timeout: 15}

  - ref: bloquer
    action: pare_feu.bloquer_ip
    depends_on: [bilan, enrichir]
    run_if: ${{ not ACTIONS.enrichir.result.interne }}
    args:
      ip: ${{ TRIGGER.ip }}

  - ref: desactiver
    action: comptes.desactiver
    depends_on: [bilan, enrichir]
    run_if: ${{ not ACTIONS.enrichir.result.interne && FN.length(ACTIONS.bilan.result.comptes_entres) > 0 }}
    args:
      compte: ${{ ACTIONS.bilan.result.comptes_entres[0] }}
    approbation: true

returns:
  verdict: >-
    ${{ 'faux positif' if ACTIONS.enrichir.result.interne && ACTIONS.bilan.result.echecs <= 3 else 'incident' }}
  rapport: >-
    ${{ TRIGGER.ip }} (int

## 1. Les expressions `${{ ... }}`

Les étapes se parlent par des **expressions** :
- `TRIGGER` : l'alerte qui a déclenché le playbook (`TRIGGER.ip`) ;
- `ACTIONS.<ref>.result` : le résultat d'une étape précédente (`ACTIONS.bilan.result.echecs`) ;
- `FN.*` : quelques fonctions (`FN.length`, `FN.join`) ; `SECRETS.<nom>` : un secret.

**Ce qu'on fait :** évaluer quelques expressions à la main, sur un contexte inventé.

In [2]:
contexte = {"TRIGGER": {"ip": "203.0.113.77", "compte": "webadmin"},
            "ACTIONS": {"bilan": {"result": {"echecs": 21, "comptes_entres": ["webadmin"]}}}}

for texte in ["${{ TRIGGER.ip }}",
              "${{ ACTIONS.bilan.result.echecs > 5 && FN.length(ACTIONS.bilan.result.comptes_entres) > 0 }}",
              "L'IP ${{ TRIGGER.ip }} est entrée sur ${{ ACTIONS.bilan.result.comptes_entres[0] }}",
              "${{ ACTIONS.bilan.result.champ_absent }}"]:
    print(f"{texte:<95} → {moteur.evaluer(texte, contexte)!r}")

${{ TRIGGER.ip }}                                                                               → '203.0.113.77'
${{ ACTIONS.bilan.result.echecs > 5 && FN.length(ACTIONS.bilan.result.comptes_entres) > 0 }}    → True
L'IP ${{ TRIGGER.ip }} est entrée sur ${{ ACTIONS.bilan.result.comptes_entres[0] }}             → "L'IP 203.0.113.77 est entrée sur webadmin"
${{ ACTIONS.bilan.result.champ_absent }}                                                        → None


❓ **Question pour la classe :** la première expression renvoie un texte, la deuxième un **booléen**. Pourquoi est-ce important de garder le type ?

*Réponse : un `run_if` a besoin d'un vrai `True`/`False`, une action `hote.tuer_processus` d'un vrai entier. Quand l'expression est seule dans la valeur, le moteur garde son type ; quand elle est au milieu d'un texte, il la convertit en texte.*

## 2. Pourquoi pas `eval()` ?

Le plus simple serait d'écrire `eval(code)`. Mais d'où viennent les données ? Le playbook est écrit par l'équipe… et **l'alerte vient de l'attaquant** (une IP, un nom d'utilisateur, une URL). Un champ d'alerte qui finit dans une expression, ou un playbook partagé et modifié, et `eval` exécute n'importe quel code sur le serveur du SOC : c'est une **injection de gabarit** (*template injection*).

**Ce qu'on fait :** notre évaluateur lit l'expression comme un **arbre** (module `ast`) et n'accepte qu'une liste blanche de constructions.

In [3]:
for texte in ["${{ __import__('os').system('id') }}",
              "${{ TRIGGER.__class__.__mro__ }}",
              "${{ open('/etc/passwd').read() }}",
              "${{ [x for x in TRIGGER] }}"]:
    try:
        moteur.evaluer(texte, contexte)
    except moteur.ExpressionInterdite as e:
        print("Refusé :", e)

Refusé : interdit dans une expression : __import__('os').system('id')
Refusé : interdit dans une expression : TRIGGER.__class__.__mro__
Refusé : interdit dans une expression : open('/etc/passwd').read()
Refusé : interdit dans une expression : [x for x in TRIGGER]


❓ **Question pour la classe :** ouvrez `_calculer` dans `moteur.py`. Pourquoi les attributs qui commencent par `_` sont-ils interdits ?

*Réponse : en Python, tout objet mène à tout le reste par ses attributs « dunder » (`__class__`, `__globals__`, `__builtins__`…). C'est le chemin classique pour sortir d'un bac à sable. Liste blanche, pas liste noire.*

## 3. Valider avant d'exécuter

**Ce qu'on fait :** un playbook faux doit être refusé **au chargement**, pas à 3 h du matin au milieu d'un incident.

In [4]:
for faux in [{"actions": [{"ref": "a", "action": "pare_feu.tout_bloquer"}]},
             {"actions": [{"ref": "a", "action": "hote.processus", "depends_on": ["b"]},
                          {"ref": "b", "action": "hote.processus", "depends_on": ["a"]}]},
             {"actions": [{"ref": "Bloquer IP", "action": "pare_feu.bloquer_ip"}]}]:
    try:
        moteur.valider(faux)
    except moteur.PlaybookInvalide as e:
        print("Invalide :", e)

Invalide : a : action inconnue « pare_feu.tout_bloquer »
Invalide : cycle entre les étapes : ['a', 'b']
Invalide : ref invalide : « Bloquer IP » (minuscules, chiffres, _)


Pour l'ordre des étapes, le moteur fait un **tri topologique** (algorithme de Kahn) sur `depends_on` : une étape ne démarre que lorsque celles dont elle dépend sont faites, et un cycle est détecté.

## 4. Exécuter le playbook sur l'intrusion

**Ce qu'on fait :** l'alerte A06 de 08:41 (connexion réussie après des échecs). Pour l'instant, l'étape qui demande une approbation est acceptée automatiquement (l'exercice 5 rendra la parole à l'analyste).

In [5]:
pb = moteur.charger("connexion-suspecte")
alerte = alertes[5]
env.a_l_heure(alerte)
ex = moteur.executer(pb, alerte)
print(ex.sortie["verdict"], "—", ex.sortie["rapport"])
moteur.tableau(ex)

incident — 203.0.113.77 (interne : False) a fait 21 échec(s) sur admin, oracle, root, webadmin, premier essai à Dec 10 07:05:12. Comptes où elle est entrée : webadmin. Blocage : réussie. Désactivation : réussie.


,ref,action,statut,essais,duree,error,result
0,bilan,journaux.connexions,réussie,1,0.03,None,"{'echecs': 21, 'succes': 1, 'comptes_essayes':..."
1,enrichir,ip.enrichir,réussie,1,0.00,None,"{'ip': '203.0.113.77', 'interne': False, 'docu..."
2,bloquer,pare_feu.bloquer_ip,réussie,1,0.00,None,"{'ip': '203.0.113.77', 'statut': 'bloquée'}"
3,desactiver,comptes.desactiver,réussie,1,0.00,None,"{'compte': 'webadmin', 'statut': 'désactivé'}"


👀 **Ce qu'on doit voir :** 4 étapes réussies, verdict *incident*, l'IP bloquée et `webadmin` désactivé. Le rapport cite le **premier essai à 07:05**, une heure et demie avant l'alerte : c'est l'étape `bilan` qui l'a trouvé.

**Ce qu'on fait :** le même playbook sur l'alerte A03 de 08:02 (l'administratrice qui se trompe de mot de passe).

In [6]:
alerte = alertes[2]
env.a_l_heure(alerte)
ex = moteur.executer(pb, alerte)
print(ex.sortie["verdict"], "—", ex.sortie["rapport"])
moteur.tableau(ex)

faux positif — 10.0.0.5 (interne : True) a fait 2 échec(s) sur jtremblay, premier essai à Dec 10 08:02:10. Comptes où elle est entrée : jtremblay. Blocage : ignorée. Désactivation : ignorée.


,ref,action,statut,essais,duree,error,result
0,bilan,journaux.connexions,réussie,1,0.02,None,"{'echecs': 2, 'succes': 1, 'comptes_essayes': ..."
1,enrichir,ip.enrichir,réussie,1,0.00,None,"{'ip': '10.0.0.5', 'interne': True, 'documenta..."
2,bloquer,pare_feu.bloquer_ip,ignorée,0,0.00,condition run_if fausse,None
3,desactiver,comptes.desactiver,ignorée,0,0.00,condition run_if fausse,None


❓ **Question pour la classe :** pourquoi `bloquer` et `desactiver` sont-elles « ignorées » ? Où est écrite cette décision ?

*Réponse : leur `run_if` est faux, car `ip.enrichir` a dit que 10.0.0.5 est **interne**. La décision est dans le playbook, lisible par toute l'équipe, pas cachée dans du code. Et même si le playbook se trompait, l'action `pare_feu.bloquer_ip` refuserait (garde-fou de l'exercice 1).*

## 5. Un playbook n'a pas de mémoire

**Ce qu'on fait :** à 09:05, l'alerte A07 (connexion sortante) lance `activite-hote.yaml`, qui arrête le mineur. Puis à 09:20 arrive l'alerte CPU (A11).

In [7]:
pb_hote = moteur.charger("activite-hote")
for alerte in (alertes[6], alertes[10]):
    env.a_l_heure(alerte)
    ex = moteur.executer(pb_hote, alerte)
    print(f"{alerte['id']} {alerte['horodatage'][11:]} → {ex.sortie['verdict']}")

A07 09:05:00 → incident
A11 09:20:00 → faux positif


❓ **Question pour la classe :** l'alerte de 09:20 est classée *faux positif*. Est-ce juste ? Pourquoi le playbook se trompe-t-il ?

*Réponse : non, c'est bien l'incident, déjà confiné à 09:05. Mais chaque exécution part de zéro : le playbook ne sait pas qu'un autre a déjà arrêté le mineur. Il faut une **mémoire** qui relie les alertes : ce sont les **cas** de l'exercice 5.*

## 6. À vous : le playbook de force brute

Les robots qui font des centaines d'essais déclenchent la règle R1. Complétez les deux `TODO` de **`playbooks/force-brute.yaml`** :
1. une étape `bloquer` (action `pare_feu.bloquer_ip`) qui attend `bilan` et ne s'exécute **que si** l'IP n'a jamais réussi à se connecter ;
2. un verdict *attaque sans impact* si aucune connexion réussie, sinon *incident*.

Le test vérifie un robot (A01) et l'attaquant, qui, lui, **est** entré.

In [ ]:
pb = moteur.charger("force-brute")
scenario.construire()
robot, attaquant = alertes[0], dict(alertes[0], ip="203.0.113.77", horodatage="2016-12-10T09:00:00")
env.a_l_heure(robot);      r1 = moteur.executer(pb, robot)
env.a_l_heure(attaquant);  r2 = moteur.executer(pb, attaquant)
assert r1.sortie["verdict"] == "attaque sans impact" and r1.historique["bloquer"]["statut"] == "réussie", moteur.tableau(r1)
assert r2.sortie["verdict"] == "incident" and r2.historique["bloquer"]["statut"] == "ignorée", moteur.tableau(r2)
print("✅ Votre playbook est correct.")

*Réponse : `solutions/playbooks/force-brute.yaml`. Pourquoi ne pas bloquer l'attaquant qui est entré ? Le bloquer ici, dans un playbook « robot », cacherait un incident grave sous une étiquette « sans impact ». Il vaut mieux le laisser au playbook d'intrusion, qui désactive aussi le compte.*

## Ce qu'il faut retenir
- Un playbook est une **donnée** (YAML) : l'équipe le lit, le relit, le versionne, sans toucher au moteur.
- Les étapes se passent leurs résultats par des **expressions** ; on les évalue avec une **liste blanche**, jamais avec `eval`, car les données viennent de l'attaquant.
- Le moteur **valide** avant d'exécuter (actions connues, pas de cycle) et garde l'**historique** de chaque étape.
- Un playbook seul n'a **pas de mémoire** entre deux alertes.
- Prochain exercice : recevoir les alertes par une **API**, et rejouer toute la matinée.